In [1]:
from test.test_align import solver_order

import diffrax
import equinox as eqx
import jax.numpy as jnp
import jax.random as jrandom
import jax.tree_util as jtu
from jax import config


config.update("jax_enable_x64", True)


def _squareplus(x):
    return 0.5 * (x + jnp.sqrt(x**2 + 4))


def sde_strong_order(solver_ctr, commutative):
    key = jrandom.PRNGKey(5678)
    driftkey, diffusionkey, ykey, bmkey = jrandom.split(key, 4)
    num_samples = 100
    bmkeys = jrandom.split(bmkey, num=num_samples)

    if commutative:
        noise_dim = 1
    else:
        noise_dim = 5

    def drift(t, y, args):
        mlp = eqx.nn.MLP(
            in_size=3,
            out_size=3,
            width_size=8,
            depth=1,
            activation=_squareplus,
            key=driftkey,
        )
        return 0.5 * mlp(y)

    def diffusion(t, y, args):
        mlp = eqx.nn.MLP(
            in_size=3,
            out_size=3 * noise_dim,
            width_size=8,
            depth=1,
            activation=_squareplus,
            final_activation=jnp.tanh,
            key=diffusionkey,
        )
        return 0.25 * mlp(y).reshape(3, noise_dim)

    t0 = 0
    t1 = 2
    y0 = jrandom.normal(ykey, (3,), dtype=jnp.float64)

    sde = (drift, diffusion, None, y0, t0, t1, noise_dim)

    # Reference solver is always an ODE-viable solver, so its implementation has been
    # verified by the ODE tests like test_ode_order.
    if issubclass(solver_ctr, diffrax.AbstractItoSolver):
        ref_solver = diffrax.Euler()
    elif issubclass(solver_ctr, diffrax.AbstractStratonovichSolver):
        ref_solver = diffrax.Heun()
    else:
        assert False

    hs, errors, order = solver_order(
        bmkeys, sde, solver_ctr(), ref_solver, 2**-13, hs_num=7
    )
    log_errs = jnp.log2(errors)
    slopes = log_errs[:-1] - log_errs[1:]

    return order, slopes

In [2]:
def _solvers():
    # solver, commutative, order
    yield diffrax.Euler, False, "Euler"  # PASSES
    yield diffrax.EulerHeun, False, "EulerHeun"  # 0.9326
    yield diffrax.Heun, False, "Heun"  # 0.86656
    yield diffrax.ItoMilstein, False, "ItoMilstein"  # 1.0025
    yield diffrax.Midpoint, False, "Midpoint"  # 0.8659
    yield diffrax.ReversibleHeun, False, "ReversibleHeun"  # 0.8666
    yield diffrax.StratonovichMilstein, False, "StratonovichMilstein"  # 0.9331
    yield diffrax.ReversibleHeun, True, "ReversibleHeun"  # 1.3648
    yield diffrax.StratonovichMilstein, True, "StratonovichMilstein"  # PASSES

In [3]:
slopes_dict = {}
for solver, comm, name in _solvers():
    order, slopes = sde_strong_order(solver, comm)
    slopes_dict.update({name: slopes})
    pretty_slopes = ["{0:0.5f}".format(slope) for slope in slopes]
    print(f"{name} has slopes {pretty_slopes} and order {order:.5f}")

2023-11-08 12:27:14.885213: W external/xla/xla/service/gpu/runtime/support.cc:58] Intercepted XLA runtime error:
INTERNAL: CpuCallback error: EqxRuntimeError: The maximum number of steps was reached in the differential equation solver. Try increasing `diffrax.diffeqsolve(..., max_steps=...)`.

At:
  /home/andy/PycharmProjects/JAX_trial/venv2/lib/python3.10/site-packages/equinox/_errors.py(70): raises
  /home/andy/PycharmProjects/JAX_trial/venv2/lib/python3.10/site-packages/jax/_src/callback.py(257): _flat_callback
  /home/andy/PycharmProjects/JAX_trial/venv2/lib/python3.10/site-packages/jax/_src/callback.py(52): pure_callback_impl
  /home/andy/PycharmProjects/JAX_trial/venv2/lib/python3.10/site-packages/jax/_src/callback.py(188): _callback
  /home/andy/PycharmProjects/JAX_trial/venv2/lib/python3.10/site-packages/jax/_src/interpreters/mlir.py(2278): _wrapped_callback
  /home/andy/PycharmProjects/JAX_trial/venv2/lib/python3.10/site-packages/jax/_src/interpreters/pxla.py(1144): __call__
 

XlaRuntimeError: The maximum number of steps was reached in the differential equation solver. Try increasing `diffrax.diffeqsolve(..., max_steps=...)`.
-------
This error occurred during the runtime of your JAX program. Setting the environment
variable `EQX_ON_ERROR=breakpoint` is usually the most useful way to debug such errors.
(This can be navigated using most of the usual commands for the Python debugger:
`u` and `d` to move through stack frames, the name of a variable to print its value,
etc.) See also `https://docs.kidger.site/equinox/api/errors/#equinox.error_if` for more
information.


In [ ]:
pretty_slopes_dict = jtu.tree_map(lambda x: "{0:0.5f}".format(x), slopes_dict)
for key, value in pretty_slopes_dict.items():
    print(f"{key}:   {value}")